# Fase 2 - Comprension de los Datos
## Seccion 15: Reporte Consolidado de Hallazgos

**Notebook:** notebooks/11_EDA_reporte_consolidado.ipynb
**Responsable:** Sofia | **Apoyo:** Steve
**Objetivo:** Consolidar todos los hallazgos de la Fase 2 en un reporte ejecutivo, copiar figuras, y preparar insumos para Fase 3.

## Configuracion inicial

In [ ]:
import pandas as pd
import numpy as np
import os, shutil, glob, json
from datetime import datetime

OUT = 'outputs'
FIGS = os.path.join('..', 'docs', 'figures')
PROC = os.path.join('..', 'data', 'processed')
RAW = os.path.join('..', 'data', 'raw')

os.makedirs(FIGS, exist_ok=True)
print('Directorios listos')

---
## Celdas 138-142: Resumen ejecutivo

### 13 Hallazgos principales (H1-H13)

In [ ]:
hallazgos = [
    ('H1', 'Cobertura real 2018-2024', 'Los datasets cubren 2018-2024, no 2015-2024 como se documento inicialmente. A1 (Properati) solo tiene datos 2020-2021.', 'Alta'),
    ('H2', '16 datasets (8A + 8B)', 'Se confirmaron 16 datasets con esquemas heterogeneos. Grupo A: oferta de propiedades. Grupo B: macroeconomicos.', 'Alta'),
    ('H3', 'Estandarizacion de esquemas', 'Se definieron 13 columnas canonicas. Los mapeos por dataset estan documentados en 01_EDA_esquema_canonico.ipynb.', 'Alta'),
    ('H4', 'Calidad de datos variable', 'A3 (House Prediction) y A8 (UPZ) tienen pocos nulos. A1 y A2 tienen nulos significativos en area y ubicacion.', 'Media'),
    ('H5', 'Outliers de precio', 'Se identificaron outliers bajos (<$10M) y altos (>$5,000M). Distribucion sesgada a la derecha (log-normal).', 'Media'),
    ('H6', 'Segmentacion geografica', 'Bogota y Medellin concentran el volumen. Ciudades intermedias con cobertura insuficiente (<80% years).', 'Media'),
    ('H7', 'A7 refuerza Villavicencio', 'A7 (scraping) aporta 1,048 registros vs 5,372 de A1. Consolidado guardado en data/processed/.', 'Media'),
    ('H8', 'Tendencia de precios al alza', 'CAGR nacional ~X%. Crecimiento acelerado en 2021-2022. Desaceleracion en 2023-2024.', 'Alta'),
    ('H9', 'Efecto pandemia (2020-2021)', 'No hubo caida generalizada de precios. Desaceleracion temporal seguida de recuperacion.', 'Media'),
    ('H10', 'Correlacion area-precio moderada', 'Spearman ~0.5-0.7. Elasticidad ~0.5-0.8 (menos que proporcional). Relacion log-lineal.', 'Media'),
    ('H11', 'Sin multicolinealidad severa', 'No se detectaron pares de predictores con |r| >= 0.7. Rooms y bathrooms tienen correlacion moderada.', 'Baja'),
    ('H12', 'IAH en deterioro', 'El Indice de Accesibilidad Habitacional aumento X% en el periodo. Se necesitan mas salarios anuales para comprar.', 'Alta'),
    ('H13', 'Consistencia con IPVN', 'Los precios de datasets siguen tendencia similar al IPVN DANE. Desviacion promedio de +-X%.', 'Media'),
]

print('===== 13 HALLAZGOS PRINCIPALES (H1-H13) =====\n')
for hid, titulo, desc, prioridad in hallazgos:
    emoji = {'Alta': '🔴', 'Media': '🟡', 'Baja': '🟢'}.get(prioridad, '⚪')
    print(f'{emoji} {hid}: {titulo}')
    print(f'   {desc}')
    print()

### Tabla resumen de decisiones para Fase 3

In [ ]:
decisiones = [
    ('Estandarizar esquemas', 'Usar mapeos canonicos de 01_EDA para unificar los 8 datasets del Grupo A.', '01_EDA_esquema_canonico'),
    ('Filtrar precios', 'Eliminar precios <=0 y nulos. Aplicar filtro P1-P99 o winsorizacion.', '03_EDA_distribucion_precios'),
    ('Filtrar area', 'Mantener solo area entre 10-800 m2. Imputar nulos en area con mediana por tipo.', '06_EDA_analisis_area'),
    ('Estandarizar ciudades', 'Unificar nombres de ciudades (Bogota, Medellin, etc.) usando diccionario de normalizacion.', '04_EDA_analisis_geografico'),
    ('Convertir moneda', 'Convertir USD a COP en A1 usando TRM historica del periodo.', '03_EDA_distribucion_precios'),
    ('Integrar A7', 'Usar consolidado Villavicencio (A1+A7) para mejorar cobertura.', '04_EDA_analisis_geografico'),
    ('Variables macro', 'Incorporar salario minimo, IPC, tasa hipotecaria como features en modelado.', '08_EDA_geoespacial_macrovariables'),
    ('IAH como target', 'Usar IAH como variable dependiente en modelos de accesibilidad.', '09_EDA_IAH_preliminar'),
    ('Validacion IPVN', 'Usar IPVN DANE como referencia para calibrar precios de oferta vs transaccion.', '10_EDA_validacion_oficial'),
]

df_decisiones = pd.DataFrame(decisiones, columns=['Decision', 'Accion', 'Notebook_origen'])
print('--- TABLA DE DECISIONES PARA FASE 3 ---')
display(df_decisiones)
df_decisiones.to_csv(os.path.join(PROC, 'decisiones_fase_3.csv'), index=False, encoding='utf-8-sig')

### Problemas identificados por dataset

In [ ]:
problemas = [
    ('A1', 'Properati', 'Moneda mixta (COP/USD), nulos en area, cobertura solo 2020-2021, l3 incluye paises'),
    ('A2', 'FincaRaiz', 'Columnas en espanol con espacios, area puede ser construida vs total, ciudad no estandarizada'),
    ('A3', 'House Prediction', '37 columnas, mayoria binarias, muchas sin mapeo canonico, sin coordenadas'),
    ('A4', 'Real Estate Bogota', 'Solo Bogota, sin coordenadas, columnas minimas (8), muestra pequena'),
    ('A5', 'Medellin 2023', 'Solo Medellin, solo 2023, neighbourhood en ingles, cobertura limitada'),
    ('A6', 'Bogota 2023', 'Solo Bogota 2023, caracteres especiales, mismo alcance que A4'),
    ('A7', 'Scraping Villavicencio', 'Scraping propio, solo FincaRaiz, periodo limitado, posibles duplicados con A2'),
    ('A8', 'UPZ Bogota', 'Solo 32 registros, dataset complementario, no apto para modelado principal'),
    ('B1', 'IPVN DANE', 'Indice trimestral, requiere conversion a anual, cobertura limitada a ciudades principales'),
    ('B2', 'Tasa hipotecaria', 'Semanal, requiere agregacion anual, columnas en espanol'),
    ('B3', 'Salario minimo', 'Serie completa 2015-2024, columna year detectada, datos limpios'),
    ('B4', 'IPC Colombia', 'Anual, requiere limpieza de nombres de columnas'),
]

df_prob = pd.DataFrame(problemas, columns=['Dataset', 'Nombre', 'Problemas'])
print('--- PROBLEMAS IDENTIFICADOS POR DATASET ---')
display(df_prob)

### Acciones correctivas propuestas

In [ ]:
correctivas = [
    ('Conversion de moneda', 'A1', 'Usar TRM historica (promedio monthly) para convertir USD a COP. Columna currency indica moneda.'),
    ('Imputacion de area', 'A1, A2', 'Imputar area nula con mediana por tipo de propiedad y ciudad. Si no hay tipo, usar mediana global.'),
    ('Filtro geografico', 'A1', 'Filtrar solo registros con l1=="Colombia" para eliminar propiedades de otros paises.'),
    ('Estandarizacion de ciudades', 'A1, A2, A7', 'Crear diccionario de normalizacion (Bogota, Bogotá, BOGOTA -> Bogota). Unificar con A5/A6/A7.'),
    ('Deteccion de duplicados', 'A2, A7', 'Detectar y eliminar duplicados intra-dataset e inter-dataset usando direccion + precio + area.'),
    ('Filtro de outliers', 'Todos Grupo A', 'Aplicar filtro IQR o percentiles (P1-P99) en precio y area. Winsorizacion opcional.'),
    ('Correccion de tipos', 'A2, A5, A6', 'Normalizar nombres de tipos de propiedad a espanol (House -> casa, Apartment -> apartamento).'),
    ('Coord geo de ciudades', 'A4, A6, A7', 'Asignar lat/lon por ciudad usando geocodificacion inversa (centroide de ciudad).'),
]

df_corr = pd.DataFrame(correctivas, columns=['Accion', 'Dataset', 'Detalle'])
print('--- ACCIONES CORRECTIVAS PROPUESTAS ---')
display(df_corr)
df_corr.to_csv(os.path.join(PROC, 'acciones_correctivas_fase_3.csv'), index=False, encoding='utf-8-sig')

---
## Celdas 143-147: Guardado de resultados

### Copiar figuras a docs/figures/

In [ ]:
print('--- COPIANDO FIGURAS A docs/figures/ ---')
png_files = glob.glob(os.path.join(OUT, '*.png'))
for f in png_files:
    dst = os.path.join(FIGS, os.path.basename(f))
    shutil.copy2(f, dst)
    print(f'  Copiado: {os.path.basename(f)}')
print(f'\nTotal figuras copiadas: {len(png_files)}')

### Copiar CSV intermedios a data/processed/

In [ ]:
print('--- COPIANDO CSV INTERMEDIOS A data/processed/ ---')
csv_files = glob.glob(os.path.join(OUT, '*.csv'))
for f in csv_files:
    dst = os.path.join(PROC, os.path.basename(f))
    shutil.copy2(f, dst)
    print(f'  Copiado: {os.path.basename(f)}')
print(f'\nTotal CSV copiados: {len(csv_files)}')

### Crear archivo HALLAZGOS_FASE_2.md

In [ ]:
hallazgos_md = '''# Hallazgos de Fase 2 - Comprension de los Datos

**Fecha de generacion:** ''' + datetime.now().strftime('%Y-%m-%d %H:%M') + '''

## Resumen Ejecutivo

Se completaron 10 notebooks de analisis exploratorio cubriendo 12 secciones (2-12 y 14).
Se identificaron 16 datasets, se estandarizaron esquemas, se analizo calidad, distribuciones,
tendencias geograficas y temporales, y se calculo el IAH preliminar.

## 13 Hallazgos Principales

'''

for hid, titulo, desc, prioridad in hallazgos:
    hallazgos_md += f'### {hid}: {titulo} ({prioridad})\n{desc}\n\n'

hallazgos_md += '''## Decisiones para Fase 3

| Decision | Accion | Notebook |
|----------|--------|----------|
'''
for d in decisiones:
    hallazgos_md += f'| {d[0]} | {d[1]} | {d[2]} |\n'

hallazgos_md += '''\n## Problemas por Dataset

| Dataset | Problemas |
|---------|----------|
'''
for p in problemas:
    hallazgos_md += f'| {p[0]} ({p[1]}) | {p[2]} |\n'

md_path = os.path.join('..', 'docs', 'HALLAZGOS_FASE_2.md')
with open(md_path, 'w', encoding='utf-8') as f:
    f.write(hallazgos_md)
print(f'HALLAZGOS_FASE_2.md creado en {md_path}')
print(hallazgos_md[:500])

### Exportar reporte de calidad consolidado

In [ ]:
print('--- REPORTE DE CALIDAD CONSOLIDADO ---')

quality_rows = []
a_files = [('A1','A1_colombia_housing_properties.csv'),('A2','A2_fincaraiz_colombia.csv'),
           ('A3','A3_colombia_house_prediction.csv'),('A4','A4_real_estate_bogota.csv'),
           ('A5','A5_medellin_properties_2023.csv'),('A6','A6_real_estate_bogota_2023.csv'),
           ('A7','A7_fincaraiz_villavicencio_scraping.csv'),('A8','A8_carac_pre_viv_nueva.csv')]
b_files = [('B1','B1_indices_precios_vivienda.csv'),('B2','B2_tasa_hipotecaria_semanal.csv'),
           ('B3','B3_salario_minimo_historico.csv'),('B4','B4_ipc_colombia_anual.csv'),
           ('B5','B5_geih_empleo_colombia.csv'),('B6','B6_qcon_confianza_constructora.csv'),
           ('B7','B7_qcon_licencias_construccion.csv'),('B8','B8_geo_estados_localidades.csv')]

for fid, fname in a_files + b_files:
    fpath = os.path.join(RAW, fname)
    if not os.path.exists(fpath):
        continue
    size_mb = os.path.getsize(fpath) / 1e6
    df = pd.read_csv(fpath, encoding='utf-8-sig', low_memory=False, nrows=1)
    n_cols = len(df.columns)
    n_rows = sum(1 for _ in open(fpath, 'r', encoding='utf-8-sig')) - 1
    
    quality_rows.append({
        'Dataset': fid,
        'Archivo': fname,
        'Filas': n_rows,
        'Columnas': n_cols,
        'Tamanio_MB': round(size_mb, 2),
        'Grupo': 'A' if fid.startswith('A') else 'B'
    })

df_quality = pd.DataFrame(quality_rows).sort_values('Dataset')
df_quality.to_csv(os.path.join(PROC, 'reporte_calidad_datasets.csv'), index=False, encoding='utf-8-sig')
display(df_quality)
print(f'\nReporte guardado en {PROC}/reporte_calidad_datasets.csv')

### Actualizar metadatos del proyecto

In [ ]:
metadatos = {
    'proyecto': 'Accesibilidad de Vivienda en Colombia',
    'fase': 2,
    'fecha_ejecucion': datetime.now().strftime('%Y-%m-%d %H:%M'),
    'notebooks': [
        '01_EDA_esquema_canonico.ipynb',
        '02_EDA_calidad_datos.ipynb',
        '03_EDA_distribucion_precios.ipynb',
        '04_EDA_analisis_geografico.ipynb',
        '05_EDA_evolucion_temporal.ipynb',
        '06_EDA_analisis_area.ipynb',
        '07_EDA_variables_categoricas.ipynb',
        '08_EDA_geoespacial_macrovariables.ipynb',
        '09_EDA_IAH_preliminar.ipynb',
        '10_EDA_validacion_oficial.ipynb',
        '11_EDA_reporte_consolidado.ipynb',
    ],
    'total_hallazgos': 13,
    'decisiones_fase_3': len(decisiones),
    'acciones_correctivas': len(correctivas),
}

meta_path = os.path.join(PROC, 'metadatos_fase_2.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadatos, f, indent=2, ensure_ascii=False)
print(f'Metadatos guardados en {meta_path}')
print(json.dumps(metadatos, indent=2, ensure_ascii=False))

---
## Resumen Final de Fase 2

### Lo que se logro:
- 11 notebooks de analisis exploratorio.
- 10 datasets del Grupo A explorados en detalle (esquemas, calidad, distribuciones).
- 6 datasets del Grupo B cargados y documentados (macrovariables).
- Esquema canonico de 13 columnas definido y mapeado.
- IAH preliminar calculado (2018-2024).
- Validacion cruzada con IPVN DANE.
- Reporte de hallazgos y decisiones documentado.

### Insumos para Fase 3 (Preparacion de Datos):
- data/processed/acciones_correctivas_fase_3.csv
- data/processed/decisiones_fase_3.csv
- data/processed/reporte_calidad_datasets.csv
- data/processed/villavicencio_consolidado.csv
- docs/HALLAZGOS_FASE_2.md
- docs/figures/ (todas las graficas)